# Visual building evidence

This notebook keeps orthophoto roof evidence and facade characterization separate from measured LiDAR geometry and reconstruction. It starts with source inspection; computer-vision features will be introduced incrementally and must retain source provenance.

**Project flow:** notebook 01 inspects LiDAR and pre-routing geometry; notebook 02 benchmarks reconstruction; this notebook isolates image-derived evidence so later fusion cannot confuse visual inference with measured height.


## 1. Load reproducible visual sources

The current notebook consumes the PNOA and Catastro preview artifacts. Acquisition remains provider code outside the notebook.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from urbanstock3d.vision.common import load_roof_objects_config, load_taxonomy
from urbanstock3d.vision.roof_objects import audit_rid2, load_rid2_source

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Change only this 14-character cadastral root to inspect another building.
CADASTRAL_ROOT_ID = "4531917YJ2743B"
assert len(CADASTRAL_ROOT_ID) == 14, "Expected a 14-character cadastral root"
BUILDING_ID = f"ES.SDGC.BU.{CADASTRAL_ROOT_ID}"
BUILDING_ROOT = PROJECT_ROOT / "outputs" / BUILDING_ID
PNOA_IMAGE_PATH = BUILDING_ROOT / "pnoa_preview.jpg"
PNOA_AUDIT_PATH = BUILDING_ROOT / "pnoa_audit.json"
FACADE_IMAGE_PATH = BUILDING_ROOT / "facade_preview.jpg"
FACADE_AUDIT_PATH = BUILDING_ROOT / "facade_audit.json"
VISION_CONFIG_ROOT = PROJECT_ROOT / "config" / "vision"

## Vision task contract

The notebook reads the same canonical taxonomy and validated task configuration as the CLI. It does not redefine classes or model parameters. V0 establishes contracts only; predictions appear after a dataset and trained model pass their verifiers.


In [ ]:
taxonomy = load_taxonomy(VISION_CONFIG_ROOT / "taxonomy.yaml")
roof_objects_config = load_roof_objects_config(VISION_CONFIG_ROOT / "roof_objects.yaml")
roof_objects_config.validate_taxonomy(taxonomy)
print(f"Task: {roof_objects_config.task_name} ({roof_objects_config.task_type})")
print(f"Taxonomy {taxonomy.version}: {taxonomy.roof_objects}")
print(f"Tracking experiment: {roof_objects_config.tracking.experiment}")
print("Inference status: unavailable until a verified model artifact exists")

### RID2 source audit

This invokes the production audit instead of duplicating dataset rules in the notebook. RID2 is open for download and local experimentation; commercial reuse remains unconfirmed because its licence field is empty. Missing canonical classes are coverage gaps, not invented label mappings.


In [ ]:
rid2_source = load_rid2_source(VISION_CONFIG_ROOT / "datasets" / "rid2.yaml")
rid2_audit = audit_rid2(rid2_source, taxonomy)
display(rid2_audit.model_dump())

### Canonical RID2 conversion review

The conversion is implemented in the core and writes a reproducible manifest plus a visual montage. This cell only displays those artifacts; it does not duplicate conversion logic.


In [ ]:
rid2_root = PROJECT_ROOT / "data" / "vision" / "rid2"
rid2_manifest_path = rid2_root / "yolo" / "manifest.json"
rid2_montage_path = rid2_root / "sample_montage.png"
if rid2_manifest_path.exists() and rid2_montage_path.exists():
    display(json.loads(rid2_manifest_path.read_text(encoding="utf-8")))
    with Image.open(rid2_montage_path) as montage:
        plt.figure(figsize=(18, 12))
        plt.imshow(montage)
        plt.axis("off")
        plt.title("RID2 canonical roof-object labels")
        plt.show()
else:
    print("Run scripts/vision/convert_rid2.py and render_rid2_montage.py first.")

### First learned predictions

This baseline uses 15 epochs and the complete RID2 training split. It establishes a reproducible benchmark; domain adaptation to Spanish PNOA imagery and footprint-constrained inference still remain.


In [ ]:
baseline_root = PROJECT_ROOT / "models" / "roof_objects" / "yolo26s-seg-rid2-baseline"
baseline_metrics_path = baseline_root / "evaluation" / "metrics.json"
baseline_montage_path = baseline_root / "evaluation" / "prediction_montage.jpg"
if baseline_metrics_path.exists() and baseline_montage_path.exists():
    display(json.loads(baseline_metrics_path.read_text(encoding="utf-8")))
    with Image.open(baseline_montage_path) as montage:
        plt.figure(figsize=(18, 12))
        plt.imshow(montage)
        plt.axis("off")
        plt.title("YOLO26s-seg RID2 baseline predictions")
        plt.show()
else:
    print("Run the RID2 baseline training and evaluation scripts first.")

In [ ]:
required_paths = [PNOA_IMAGE_PATH, PNOA_AUDIT_PATH, FACADE_IMAGE_PATH, FACADE_AUDIT_PATH]
missing_paths = [path for path in required_paths if not path.exists()]
assert not missing_paths, f"Generate the visual evidence first: {missing_paths}"

pnoa_audit = json.loads(PNOA_AUDIT_PATH.read_text(encoding="utf-8"))
facade_audit = json.loads(FACADE_AUDIT_PATH.read_text(encoding="utf-8"))
with Image.open(PNOA_IMAGE_PATH) as image:
    pnoa_image = np.asarray(image.convert("RGB"))
with Image.open(FACADE_IMAGE_PATH) as image:
    facade_image = np.asarray(image.convert("RGB"))

print(f"PNOA shape: {pnoa_image.shape}; pixel size: {pnoa_audit['request']['pixel_size_m']} m")
print(f"Facade shape: {facade_image.shape}; status: {facade_audit['status']}")

## 2. Inspect orthophoto and facade evidence


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(pnoa_image)
axes[0].set_title("PNOA orthophoto — roof evidence")
axes[1].imshow(facade_image)
axes[1].set_title("Catastro facade — visual characterization")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Planned computer-vision evidence

Next additions: image validity and sharpness, shadow and vegetation proxies, structural edge candidates, roof-object evidence, facade openings and material hypotheses. Image edges must never be treated automatically as roof structure.


## Observations

Record acquisition quality, occlusions, shadows, vegetation, temporal mismatch, and which visual cues are defensible for this building.
